In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 50)   # показывать все колонки
sns.set_style('whitegrid')                 # приятный стиль графиков
print("Библиотеки загружены")

Библиотеки загружены


In [10]:
df = pd.read_csv('../data/contracts.csv', low_memory=False)
print("Размер датасета:", df.shape)
df.head(3)

Размер датасета: (887642, 39)


,Winner,Winner_VAT,Winner_Country,Winner_City,Winner_Address,Type,Contract_Type,Procedure_Type,Contracting_Authority,Contracting_Authority_VAT,Contracting_Authority_Type,Contracting_Authority_Activity_Type,Award_Anouncement_Number,Award_Announcement_Date,Contract_Conclusion_Type,Award_Criteria_Type,With_Electronic_Auction,Offers_Number,Subcontracted,Contract_Number,Contract_Date,Contract_Title,Value,Currency,Value_RON,Value_EUR,CPV_Code_ID,CPV_Code,Participation_Announcement_Number,Participation_Announcement_Date,Participation_Estimated_Value,Participation_Estimated_Value_Currency,EU_Funds,Financing_Type,Legislation_Type_ID,EU_Fund,Periodic_Contract,Garantee_Deposits,Financing_Method
0,Top Diagnostics S.R.L.,10572840,Romania,Bucuresti,Str Orlando Nr 6 Sector 1,Anunt de atribuire la anunt de participare,Furnizare,Licitatie deschisa,MINISTERUL APARARII UM 02534 IASI,4540054,Altele: Minister sau orice alta autor nati. sa...,Sanatate,10745.0,2007-09-17 10:18:49.187,Un contract de achizitii publice,Pretul cel mai scazut,NaN,10.0,NaN,A/1605/1,2007-08-23 00:00:00.000,CONTRACT FURNIZARE REACTIVI,1568.50,RON,1568.50,480.27,1881.0,24496500-2,20796.0,2007-06-26 14:58:28.587,89293.00,EUR,NaN,NaN,NaN,NaN,NaN,Garantiile de participare in valoare de 1000 R...,CASAOPSNAJ
1,A&A MEDICAL SRL SUCURSALA IASI,11461697,Romania,IASI,"STR. PROF. I. SIMIONESCU, BL. SD9",Anunt de atribuire la anunt de participare,Furnizare,Licitatie deschisa,MINISTERUL APARARII UM 02534 IASI,4540054,Altele: Minister sau orice alta autor nati. sa...,Sanatate,10745.0,2007-09-17 10:18:49.187,Un contract de achizitii publice,Pretul cel mai scazut,NaN,10.0,NaN,A/1605/2,2007-08-23 00:00:00.000,CONTRACT FURNIZARE REACTIVI,2389.80,RON,2389.80,731.74,1881.0,24496500-2,20796.0,2007-06-26 14:58:28.587,89293.00,EUR,NaN,NaN,NaN,NaN,NaN,Garantiile de participare in valoare de 1000 R...,CASAOPSNAJ
2,SC POLISANO SRL,4101148,Romania,Sibiu,"Str. 9 MAI, Bl.77",Anunt de atribuire la anunt de participare,Furnizare,Licitatie deschisa,SPITALUL MUNICIPAL PLOIESTI,2844227,NaN,Sanatate,17129.0,2007-11-28 01:30:00.800,Un contract de achizitii publice,Pretul cel mai scazut,NaN,4.0,NaN,5955,2007-11-23 00:00:00.000,Achizitie medicamente antineoplazice,3738.26,RON,3738.26,1040.02,1837.0,24452100-8,32520.0,2007-09-15 01:30:00.217,526376.14,RON,NaN,NaN,NaN,NaN,NaN,Se cer garantii de participare doar pentru:Lot...,Fonduri primite de la Casa de Asigurari de San...


In [ ]:
# single_bidder = 1, если в тендере был ровно один участник (Offers_Number == 1)
# Строки, где Offers_Number пустой (34 шт.), убираем — без таргета они бесполезны
df = df.dropna(subset=['Offers_Number']).copy()
df['single_bidder'] = (df['Offers_Number'] == 1).astype(int)

print("Распределение классов:")
print(df['single_bidder'].value_counts())
print("\nДоля single-bidder тендеров:",
      round(df['single_bidder'].mean() * 100, 1), "%")

In [ ]:
leakage_cols = [
    'Winner', 'Winner_VAT', 'Winner_Country', 'Winner_City', 'Winner_Address',
    'Value', 'Value_RON', 'Value_EUR',
    'Award_Anouncement_Number', 'Award_Announcement_Date',
    'Contract_Number', 'Contract_Date', 'Contract_Title',
    'Contract_Conclusion_Type', 'Subcontracted',
    'Offers_Number',
]


In [ ]:
high_null_cols = [
    'Periodic_Contract', 'EU_Fund', 'EU_Funds', 'With_Electronic_Auction',
    'Financing_Type', 'Garantee_Deposits', 'Financing_Method',
    'Contracting_Authority_Type', 'Legislation_Type_ID',
    'Participation_Announcement_Number', 'Participation_Announcement_Date',
    'CPV_Code_ID',
    'Contracting_Authority_VAT',
    'Participation_Estimated_Value', 'Participation_Estimated_Value_Currency',
]

cols_to_drop = leakage_cols + high_null_cols
df_clean = df.drop(columns=cols_to_drop)
print("Колонок было:", df.shape[1])
print("Колонок осталось:", df_clean.shape[1])
print("\nОставшиеся колонки:")
print(df_clean.columns.tolist())


In [ ]:
df_sample = df_clean.sample(n=150_000, random_state=42).reset_index(drop=True)
print("Размер рабочей выборки:", df_sample.shape)
print("Доля single-bidder в выборке:",
      round(df_sample['single_bidder'].mean() * 100, 1), "%")


In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df_sample, x='single_bidder',
              hue='single_bidder', palette=['#4C72B0', '#DD8452'], legend=False)
plt.title('Распределение классов: single-bidder vs конкурентные')
plt.xlabel('single_bidder (0 = конкурентный, 1 = единственный участник)')
plt.ylabel('Количество тендеров')
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=120)
plt.show()

In [ ]:
proc_stats = (df_sample.groupby('Procedure_Type')['single_bidder']
              .agg(['mean', 'count'])
              .sort_values('mean', ascending=False))
proc_stats['mean'] = (proc_stats['mean'] * 100).round(1)
proc_stats.columns = ['Доля single-bidder %', 'Кол-во тендеров']
print(proc_stats)


In [ ]:
print("Пропуски в рабочей выборке:")
nulls = df_sample.isnull().sum()
nulls = nulls[nulls > 0].sort_values(ascending=False)
print(nulls if len(nulls) else "Пропусков нет")

In [29]:
df_sample.to_csv('tenders_clean.csv', index=False)
print("Готово! Сохранено в tenders_clean.csv")
print("Размер:", df_sample.shape)


Готово! Сохранено в tenders_clean.csv
Размер: (150000, 9)
